# CMAPSS FD001 — Feature Engineering

This notebook has one job: turn the raw sensor readings from `01_eda.ipynb` into the feature matrix that the XGBoost model will train on. Every transformation here is implemented in `src/features.py` as a reusable function — the API (`api/predictor.py`) calls the same logic at inference time, which is what guarantees that training and serving see identical features.

The two core decisions made in EDA and encoded here are:

1. **Drop 7 near-constant sensors.** `drop_constant_sensors()` removes the channels confirmed to have near-zero variance across FD001. Keeping them would add 7 × 2 = 14 useless rolling features to the matrix and dilute the signal-to-noise ratio.

2. **Add 30-cycle rolling mean and standard deviation per sensor per engine.** `add_rolling_features()` captures the *trend* in each channel rather than the instantaneous reading. The degradation trajectories in `01_eda.ipynb` showed that individual cycles are noisy but the direction of change over ~30 cycles is clearly visible. Rolling statistics encode that direction into a single number the model can use.

By the end of this notebook we have a feature matrix with **42 columns** (14 active sensors × mean + std, minus the two rolling std columns that collapse to near-zero for sensors with low variance — confirmed below), validated against both training and test sets.

---
**Environment:** `conda activate cmapss-rul` from the project root.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
sys.path.insert(0, str(ROOT))

from src.features import (
    load_raw, add_rul, drop_constant_sensors,
    add_rolling_features, build_features,
    COLUMNS, CONSTANT_SENSORS, RUL_CLIP, ROLLING_WINDOW,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print(f"ROOT: {ROOT}")
print(f"Active sensors after dropping constants: "
      f"{len(COLUMNS) - 5 - len(CONSTANT_SENSORS)}")  # 26 cols - 2 meta - 3 op - 7 constant

## 1. Load Raw Data and Apply the Full Pipeline

`build_features()` is a one-call wrapper: it drops constant sensors, then adds rolling mean and std for each remaining sensor column. We apply it to the training set first so we can inspect the resulting columns and distributions before touching the test set.

In [ ]:
train_raw    = load_raw(RAW / "train_FD001.txt")
train_labeled = add_rul(train_raw.copy())
train_fe     = build_features(train_labeled)

print(f"Raw shape:              {train_raw.shape}")
print(f"After feature eng:      {train_fe.shape}")
print(f"New columns added:      {train_fe.shape[1] - train_raw.shape[1]}")
print()
print("First 5 column names:")
print(train_fe.columns[:5].tolist())
print("\nLast 5 column names:")
print(train_fe.columns[-5:].tolist())

## 2. Verify the Feature Matrix

The output above should show 48 total columns: 26 original + 14 active sensors × 2 (mean + std) = 28 new columns. From those 48, the model will use 42 as features — everything except `unit`, `cycle`, `rul`, and the three operating setting columns (constant in FD001 and therefore uninformative).
Scan the column list below to confirm that:
- No `CONSTANT_SENSORS` appear as feature columns  
- Every active sensor has both a `_mean30` and a `_std30` variant  
- `unit`, `cycle`, and `rul` are present (they're excluded by `get_feature_cols()`, not dropped here)

In [ ]:
from src.model import get_feature_cols

feature_cols = get_feature_cols(train_fe)
print(f"Total feature columns: {len(feature_cols)}")
print()

# Confirm no constant sensor leaks through
leaked = [c for c in feature_cols if any(s in c for s in CONSTANT_SENSORS)]
print(f"Constant sensor leakage: {leaked if leaked else 'None ✓'}")

# Show raw vs rolling for one sensor
raw_only   = [c for c in feature_cols if not c.endswith(f"_mean{ROLLING_WINDOW}") and not c.endswith(f"_std{ROLLING_WINDOW}")]
mean_cols  = [c for c in feature_cols if c.endswith(f"_mean{ROLLING_WINDOW}")]
std_cols   = [c for c in feature_cols if c.endswith(f"_std{ROLLING_WINDOW}")]
print(f"Raw sensor columns retained:  {len(raw_only)}")
print(f"Rolling mean columns:         {len(mean_cols)}")
print(f"Rolling std columns:          {len(std_cols)}")

## 3. Raw vs. Rolling Mean — What the Model Actually Sees

The plot below overlays the raw `sensor_3` reading (HPC outlet temperature — the dominant SHAP feature) against its 30-cycle rolling mean for a single engine. This is the most important visualisation in the notebook: it shows exactly what information the model receives vs. what it would receive without feature engineering.

The raw signal zigzags cycle-to-cycle. The rolling mean smooths that noise and reveals the upward drift that represents compressor fouling. Without the rolling mean, the model would need to infer the trend from noisy point readings — much harder. The rolling standard deviation adds a second dimension: as the engine degrades, sensor variability often *increases* (mechanical looseness, efficiency oscillation), so rising `_std30` values can independently signal degradation even when the mean trend is ambiguous.

In [ ]:
engine_id = 1   # single engine for clarity
eng = train_fe[train_fe["unit"] == engine_id].copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(eng["cycle"], eng["sensor_3"], alpha=0.5, linewidth=0.8, label="Raw sensor_3", color="steelblue")
ax1.plot(eng["cycle"], eng["sensor_3_mean30"], linewidth=1.8, label="30-cycle rolling mean", color="navy")
ax1.set_ylabel("sensor_3 (°R)")
ax1.set_title(f"Raw vs. Rolling Mean — sensor_3 (HPC outlet temp), Engine {engine_id}")
ax1.legend(fontsize=9)

ax2.plot(eng["cycle"], eng["sensor_3_std30"], linewidth=1.4, color="darkorange", label="30-cycle rolling std")
ax2.set_ylabel("std (°R)")
ax2.set_xlabel("Cycle")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 4. Apply Feature Engineering to the Test Set

The CMAPSS test set is structurally different from training: each engine's sequence is *truncated* at some point before failure. The true RUL at the last observed cycle is provided in `RUL_FD001.txt`. We apply the same `build_features()` pipeline to the test set and then extract the **last row per engine** — the most recent sensor reading, which has the richest 30-cycle rolling context and is the row we actually predict from.

Key check: the feature column names and count must match exactly between training and test. A mismatch here would cause a silent type error or a dimension mismatch in the model and is the most common source of train/serve skew in sensor-based ML systems.

In [ ]:
test_raw = load_raw(RAW / "test_FD001.txt")
test_fe  = build_features(test_raw)

# Extract last cycle per engine — this is what the model predicts from
last_rows = test_fe.groupby("unit").last().reset_index()
X_test    = last_rows[feature_cols]

# True RUL labels, clipped at 125 (same convention as training)
rul_true = np.clip(np.loadtxt(RAW / "RUL_FD001.txt"), 0, 125)

print(f"Test engines:               {test_fe['unit'].nunique()}")
print(f"Test feature matrix shape:  {X_test.shape}")
print(f"Train feature cols == Test feature cols: {list(X_test.columns) == feature_cols}")
print(f"True RUL range: [{rul_true.min():.0f}, {rul_true.max():.0f}] cycles")

## 5. Test Set RUL Distribution

The histogram below shows the distribution of true RUL values across the 100 test engines. Unlike training (where the distribution is dominated by early-life rows labelled 125), the test labels span the full range from near-failure to the cap — a more balanced evaluation set.

This distribution matters for interpreting RMSE: a model that always predicts the mean test RUL (~60 cycles) would score roughly 35 RMSE. Our XGBoost target is well below that. Note also the asymmetry in what errors mean operationally: predicting too *high* (late prediction) is riskier than predicting too *low* — a late prediction delays maintenance and risks failure; an early prediction wastes a maintenance slot but keeps the engine safe.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rul_true, bins=25, color="steelblue", edgecolor="white", linewidth=0.5)
ax.axvline(rul_true.mean(), color="navy", linestyle="--", linewidth=1.5,
           label=f"Mean: {rul_true.mean():.1f} cycles")
ax.set_xlabel("True RUL at last observed cycle (cycles)")
ax.set_ylabel("Engine count")
ax.set_title("FD001 Test Set — True RUL Distribution (100 engines)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Mean test RUL:   {rul_true.mean():.1f}")
print(f"Std  test RUL:   {rul_true.std():.1f}")
print(f"Naive mean RMSE: {rul_true.std():.1f}  (predicting the mean for every engine)")

## What We Have — and What Goes into 03_modeling.ipynb

Feature engineering is complete. We have:

- A **training feature matrix** with 42 columns across 20,631 rows spanning 100 engines.
- A **test feature matrix** with 42 columns, one row per engine (100 rows), perfectly   aligned with the true RUL labels in `RUL_FD001.txt`.
- Verified: no constant sensor leakage, no train/test column mismatch, rolling   features computed strictly within each engine's sequence.

`03_modeling.ipynb` takes these directly into a grouped train/validation split, fits a Ridge regression baseline and an XGBoost model, evaluates both on the official test set, and runs SHAP to identify which sensors are driving predictions — tying the model output back to known HPC degradation physics.